# 03. Change Data Feed (CDF) - 変更だけを下流に流す

`01` と `02` では、ファイルをテーブルに取り込むところを扱いました。
ここからは **テーブルからテーブルへ** データを流す話になります。

困るのは、上流のテーブルが更新されたときです。
Silver層のテーブルが1行だけ更新されたとして、下流のGold層はどうやってそれを知るのでしょうか。

- 毎回テーブル全体を読み直す … 確実だが、テーブルが大きいと重い
- 変更された行だけ読む … 軽いが、「どれが変更されたか」を知る手段が要る

この2つ目を可能にするのが Change Data Feed (CDF) です。

このノートブックで確かめること:

1. Deltaテーブルは変更のたびに **バージョン** が増えること
2. CDFを有効にすると、行単位の変更履歴が読めること
3. その履歴をストリームで下流に流せること
4. CDFの制約 (いつからの変更が記録されるか)

**前提**: `00_setup` を実行済みであること。`01` の内容 (チェックポイント) を理解していること。

## 準備

In [ ]:
from databricks.connect import DatabricksSession
from databricks.sdk.errors import NotFound
from databricks.sdk.runtime import dbutils

spark = DatabricksSession.builder.profile("free").serverless(True).getOrCreate()

In [ ]:
CATALOG = "tech_survey"

# 変更が起きる側 (上流)
TABLE_SILVER = f"{CATALOG}.silver.orders"

# 変更を受け取る側 (下流)
TABLE_GOLD = f"{CATALOG}.gold.orders_changes"

CHECKPOINT = f"/Volumes/{CATALOG}/ops/checkpoints/03_cdf"

## 1. まず普通のテーブルを作る

CDFはまだ有効にしません。有効にする前と後で何が変わるかを見たいためです。

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE_SILVER}")
spark.sql(f"DROP TABLE IF EXISTS {TABLE_GOLD}")

try:
    dbutils.fs.rm(CHECKPOINT, True)
except NotFound:
    pass

spark.sql(f"""
    CREATE TABLE {TABLE_SILVER} (
        order_id INT,
        product STRING,
        amount INT,
        status STRING
    )
""")

spark.sql(f"""
    INSERT INTO {TABLE_SILVER} VALUES
        (1, 'laptop',   150000, 'placed'),
        (2, 'monitor',   40000, 'placed'),
        (3, 'keyboard',  12000, 'placed')
""")

display(spark.table(TABLE_SILVER))

## 2. Deltaのバージョンを見る

Deltaテーブルは、書き込みのたびに新しい **バージョン** を作ります。
上書きしているように見えても、内部では履歴が積み上がっています。

`DESCRIBE HISTORY` でその履歴が見られます。
`CREATE` で version 0、`INSERT` で version 1 ができているはずです。

In [ ]:
display(spark.sql(f"DESCRIBE HISTORY {TABLE_SILVER}"))

ここで見えているのは「テーブル全体に対して何をしたか」という操作の記録です。
**どの行がどう変わったか** までは分かりません。それを記録させるのがCDFです。

## 3. CDFを有効にする

テーブルのプロパティとして設定します。

重要な制約が1つあります。**有効にした後の変更しか記録されません。**
過去にさかのぼって変更履歴が作られるわけではないので、
必要になってから有効にしても、それ以前の変更は取れません。

In [ ]:
spark.sql(f"""
    ALTER TABLE {TABLE_SILVER}
    SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")

# 有効になった時点のバージョンを控えておく。ここから後の変更が記録される
version_cdf_on = spark.sql(f"DESCRIBE HISTORY {TABLE_SILVER}").select("version").first()[0]
print("CDFを有効にしたバージョン:", version_cdf_on)

## 4. データを変更する

更新・削除・追加を1つずつやります。
Deltaテーブルは、普通のファイル形式と違って `UPDATE` や `DELETE` ができます。

In [ ]:
# 1件を更新する
spark.sql(f"UPDATE {TABLE_SILVER} SET status = 'shipped' WHERE order_id = 1")

# 1件を削除する
spark.sql(f"DELETE FROM {TABLE_SILVER} WHERE order_id = 2")

# 1件を追加する
spark.sql(f"INSERT INTO {TABLE_SILVER} VALUES (4, 'mouse', 5000, 'placed')")

display(spark.table(TABLE_SILVER))

## 5. 変更履歴を読む

`table_changes(テーブル名, 開始バージョン)` で、そのバージョン以降の変更が行単位で読めます。

通常の列に加えて、次の3つが付いてきます。

- `_change_type` … その行に何が起きたか
- `_commit_version` … どのバージョンでの変更か
- `_commit_timestamp` … いつの変更か

`_change_type` には4種類の値が入ります。実行して、どの操作がどれに対応するか確かめてください。

In [ ]:
display(
    spark.sql(f"""
        SELECT * FROM table_changes('{TABLE_SILVER}', {version_cdf_on})
        ORDER BY _commit_version
    """)
)

### `update_preimage` と `update_postimage`

1件の `UPDATE` に対して、行が **2つ** 出てきたはずです。

- `update_preimage` … 更新される前の状態
- `update_postimage` … 更新された後の状態

「何が何に変わったか」を両方記録しているので、差分を追いかけられます。
一方で下流に「更新後の値」だけを流したい場合は、`update_preimage` を除外する必要があります。

`INSERT` は `insert`、`DELETE` は `delete` の1行だけです。

## 6. 変更をストリームで下流に流す

ここが実用的な使い方です。`readStream` に `readChangeFeed` を指定すると、
変更履歴をストリームのソースとして扱えます。

`01` でファイルを読んだときと同じで、チェックポイントが「どのバージョンまで流したか」を覚えます。
なので次に実行したときは、その後の変更だけが流れます。

In [ ]:
changes = (
    spark.readStream.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", version_cdf_on)
    .table(TABLE_SILVER)
)

query = (
    changes.writeStream.option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLE_GOLD)
)
query.awaitTermination()

display(spark.table(TABLE_GOLD).orderBy("_commit_version"))

## 7. もう一度変更して、流す

上流をもう1件更新してから、**まったく同じストリーム処理** を動かします。
何件が下流に流れるか、予想してから実行してください。

In [ ]:
spark.sql(f"UPDATE {TABLE_SILVER} SET amount = 999 WHERE order_id = 3")

In [ ]:
changes = (
    spark.readStream.format("delta")
    .option("readChangeFeed", "true")
    .option("startingVersion", version_cdf_on)
    .table(TABLE_SILVER)
)

query = (
    changes.writeStream.option("checkpointLocation", CHECKPOINT)
    .trigger(availableNow=True)
    .toTable(TABLE_GOLD)
)
query.awaitTermination()

print("このバッチで読んだ行数:", sum(p.numInputRows for p in query.recentProgress))
display(spark.table(TABLE_GOLD).orderBy("_commit_version"))

## 8. 注意点

CDFを使う前に知っておくべきことがあります。

**有効にした後の変更しか記録されない**

`3.` で触れたとおりです。運用中のテーブルで後から有効にしても、過去の変更は取れません。

**永久に残るわけではない**

変更履歴はテーブルの履歴と一緒に管理されていて、保持期間を過ぎた分は `VACUUM` で消えます。
下流の処理が長期間止まっていると、読もうとしたバージョンが既に消えていて読めないことがあります。
保持期間の設定は `11_vacuum_time_travel` で扱います。

**書き込みコストが増える**

変更履歴を別途書き出すため、有効にしていないテーブルよりも書き込みが重くなり、ストレージも増えます。
更新がほとんど起きないテーブルや、下流が全件読み直しでも困らない規模なら、有効にしない選択もあります。

**そのままでは「現在の状態」にならない**

下流に溜まるのは変更の記録です。`delete` や `update_preimage` も混ざっているので、
これを「今どうなっているか」の表にするには `MERGE` が要ります。`05_merge_into` で扱います。

## 考えてみる

- `7.` で下流に流れたのは何件でしたか。なぜその件数になったのでしょうか
- CDFを使わずに同じことをするなら、どんな方法が考えられますか
- Bronze層のテーブルにCDFを有効にする意味はあるでしょうか

### 答え

**Q1. `7.` で流れた件数**

2件です。`UPDATE` 1回で `update_preimage` と `update_postimage` の2行が作られるためです。

それ以前の変更が流れてこないのは、チェックポイントが「どのバージョンまで流したか」を
覚えているからです。`startingVersion` を指定していても、チェックポイントがある限りそちらが優先されます。
`01` でファイルが読み直されなかったのと同じ仕組みです。

**Q2. CDFを使わない方法**

いくつかあります。それぞれ手間と弱点があります。

- 毎回テーブル全体を読み直して下流を作り直す … 確実だが、テーブルが大きいと時間もコストもかかる
- 更新時刻の列 (`updated_at` など) を持たせて、前回実行時刻より新しい行だけ読む …
  よく使われる方法だが、**削除された行を検知できない**。列の更新漏れも起きる
- アプリケーション側で変更ログを別テーブルに書く … 自由度は高いが、書き漏れの責任を自分で負う

CDFの利点は、削除も含めてDeltaが自動で記録してくれる点にあります。

**Q3. Bronze層にCDFを有効にする意味**

多くの場合は薄いです。Bronze層は「届いたものをそのまま追記する」使い方が基本で、
更新や削除がほとんど起きません。すべてが `insert` なら、CDFを使わなくても
`01` でやったようにストリームで読めば同じことができます。書き込みコストが増えるだけ損になります。

CDFが効くのは、更新や削除が起きる層です。典型的にはSilver層から下流への伝播になります。

## 後片付け

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {TABLE_SILVER}")
spark.sql(f"DROP TABLE IF EXISTS {TABLE_GOLD}")

try:
    dbutils.fs.rm(CHECKPOINT, True)
except NotFound:
    pass